<a href="https://colab.research.google.com/github/NomadZhang/DSA5204/blob/main/03_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 Training Pipeline

This notebook runs the LoRA experiments for TinyLlama and saves structured outputs under `./results/`.

It is designed for Google Colab GPU and supports:
- a shared train/validation split
- response-only loss masking
- LoRA rank sweeps for `r = 2, 4, 8, 16`
- an optional full fine-tuning baseline attempt
- automatic logging of trainable parameters, evaluation loss, perplexity, wall-clock time, and peak GPU memory
- lightweight result saving without full checkpoints for every run


In [1]:
from pathlib import Path

repo_dir = Path('/content/DSA5204')
if not repo_dir.exists():
    !git clone https://github.com/NomadZhang/DSA5204.git /content/DSA5204

%cd /content/DSA5204
!pip install -q "transformers>=4.57.0" "datasets>=2.19.0" accelerate pandas matplotlib sentencepiece seaborn


Cloning into '/content/DSA5204'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 57 (delta 18), reused 38 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 4.70 MiB | 4.71 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/DSA5204


In [2]:
import inspect
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import Trainer, TrainingArguments, default_data_collator

from src.experiment_utils import (
    DEFAULT_SAMPLE_PROMPTS,
    alpha_for_rank,
    build_model,
    generate_samples,
    load_and_prepare_datasets,
    load_tokenizer,
    parameter_statistics,
    peak_gpu_memory_mb,
    reset_peak_gpu_memory,
    safe_perplexity,
    set_all_seeds,
    write_json,
)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATA_PATH = "./data/train.jsonl"
RESULTS_DIR = Path("./results")
SEED = 42
VALIDATION_SIZE = 0.1
MAX_LENGTH = 256
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_STEPS = 50
EVAL_STEPS = 25
LOGGING_STEPS = 10
RUN_FULL_FINE_TUNING_BASELINE = True
RANKS = [2, 4, 8, 16]
SAMPLE_PROMPTS = DEFAULT_SAMPLE_PROMPTS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = None

set_all_seeds(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Transformers version: {transformers.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory total: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")


Transformers version: 5.0.0
Device: cuda
GPU: Tesla T4
CUDA memory total: 14.56 GB


In [3]:
tokenizer = load_tokenizer(MODEL_ID)
raw_splits, tokenized_splits = load_and_prepare_datasets(
    DATA_PATH,
    tokenizer,
    max_length=MAX_LENGTH,
    validation_size=VALIDATION_SIZE,
    seed=SEED,
)

print(raw_splits)
print(tokenized_splits)
display(raw_splits["train"].select(range(3)).to_pandas())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Tokenizing with max_length=256:   0%|          | 0/1502 [00:00<?, ? examples/s]

Filtering examples with no supervised tokens:   0%|          | 0/13509 [00:00<?, ? examples/s]

Filtering examples with no supervised tokens:   0%|          | 0/1502 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['prompt', 'response'],
        num_rows: 1502
    })
})
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 1502
    })
})


,prompt,response
0,Instruction: Given the reference text below th...,Mount Vesuvius had a reduced summit by 450 met...
1,Instruction: What is Visual Studio Code?\nResp...,Visual Studio Code is a free code editor redef...
2,Instruction: Approx. how many nurses were enro...,"More than 100,000 nurses were enrolled to the ..."


In [4]:
def build_run_configs():
    configs = []
    if RUN_FULL_FINE_TUNING_BASELINE:
        configs.append(
            {
                "run_name": "full_ft_baseline",
                "method": "full_ft",
                "use_lora": False,
                "r": None,
                "alpha": None,
            }
        )

    for rank in RANKS:
        configs.append(
            {
                "run_name": f"lora_r{rank}",
                "method": "lora",
                "use_lora": True,
                "r": rank,
                "alpha": alpha_for_rank(rank),
            }
        )
    return configs


def make_training_args(run_name):
    training_kwargs = {
        "output_dir": str(RESULTS_DIR / run_name / "trainer_output"),
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "max_steps": MAX_STEPS,
        "eval_steps": EVAL_STEPS,
        "save_strategy": "no",
        "logging_steps": LOGGING_STEPS,
        "report_to": "none",
        "remove_unused_columns": False,
        "fp16": False,
        "dataloader_pin_memory": torch.cuda.is_available(),
        "seed": SEED,
    }

    signature = inspect.signature(TrainingArguments.__init__).parameters
    if "overwrite_output_dir" in signature:
        training_kwargs["overwrite_output_dir"] = True
    if "evaluation_strategy" in signature:
        training_kwargs["evaluation_strategy"] = "steps"
    elif "eval_strategy" in signature:
        training_kwargs["eval_strategy"] = "steps"
    if "bf16" in signature:
        training_kwargs["bf16"] = False

    return TrainingArguments(**training_kwargs)


def run_experiment(config):
    run_name = config["run_name"]
    run_dir = RESULTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n===== Running {run_name} =====")
    reset_peak_gpu_memory()
    set_all_seeds(SEED)

    model = None
    trainer = None
    status = "completed"
    error_message = None
    eval_metrics = {}
    train_output = None
    train_runtime_sec = 0.0
    global_step = 0
    stats = {
        "total_params": None,
        "trainable_params": None,
        "frozen_params": None,
        "trainable_ratio": None,
    }
    gpu_memory_after_load_mb = None

    try:
        model = build_model(
            MODEL_ID,
            use_lora=config["use_lora"],
            r=config["r"],
            alpha=config["alpha"],
            torch_dtype=DTYPE,
        )
        model.to(DEVICE)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()

        stats = parameter_statistics(model)
        gpu_memory_after_load_mb = torch.cuda.memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

        trainer = Trainer(
            model=model,
            args=make_training_args(run_name),
            train_dataset=tokenized_splits["train"],
            eval_dataset=tokenized_splits["test"],
            data_collator=default_data_collator,
        )

        start_time = time.time()
        train_output = trainer.train()
        train_runtime_sec = time.time() - start_time
        eval_metrics = trainer.evaluate()
        global_step = trainer.state.global_step

        model.config.use_cache = True
        samples = generate_samples(model, tokenizer, SAMPLE_PROMPTS, device=DEVICE)
        write_json(run_dir / "sample_generations.json", {"samples": samples})

    except RuntimeError as exc:
        train_runtime_sec = time.time() - start_time if 'start_time' in locals() else 0.0
        status = "oom" if "out of memory" in str(exc).lower() else "failed"
        error_message = str(exc)
        if model is not None:
            stats = parameter_statistics(model)
        gpu_memory_after_load_mb = torch.cuda.memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Run {run_name} ended with status={status}: {error_message}")

    peak_memory = peak_gpu_memory_mb()
    eval_loss = eval_metrics.get("eval_loss")
    perplexity = safe_perplexity(eval_loss) if eval_loss is not None else None
    train_loss = train_output.training_loss if train_output is not None else None
    step_time_sec = train_runtime_sec / global_step if global_step else None

    summary = {
        "run_name": run_name,
        "status": status,
        "method": config["method"],
        "r": config["r"],
        "alpha": config["alpha"],
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        **stats,
        "gpu_memory_after_load_mb": gpu_memory_after_load_mb,
        "peak_gpu_memory_mb": peak_memory,
        "train_runtime_sec": train_runtime_sec,
        "step_time_sec": step_time_sec,
        "global_step": global_step,
        "train_loss": train_loss,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "error_message": error_message,
    }

    write_json(run_dir / "metrics_summary.json", summary)

    if trainer is not None:
        del trainer
    if model is not None:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


run_summaries = [run_experiment(config) for config in build_run_configs()]
summary_df = pd.DataFrame(run_summaries)
summary_df.to_csv(RESULTS_DIR / "experiment_summary.csv", index=False)
display(summary_df)



===== Running full_ft_baseline =====


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step,Training Loss,Validation Loss
25,2.211621,2.152987
50,2.304597,2.017092


...

Step,Training Loss,Validation Loss
25,1.685832,1.666837
50,1.809431,1.637698



===== Running lora_r16 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
25,1.664621,1.630043
50,1.797996,1.614362


,run_name,status,method,r,alpha,seed,max_length,max_steps,learning_rate,gradient_accumulation_steps,...,trainable_ratio,gpu_memory_after_load_mb,peak_gpu_memory_mb,train_runtime_sec,step_time_sec,global_step,train_loss,eval_loss,perplexity,error_message
0,full_ft_baseline,completed,full_ft,NaN,NaN,42,256,50,0.0002,8,...,100.000000,2100.176758,8586.855469,1090.034738,21.800695,50,2.321505,2.017092,7.516432,None
1,lora_r2,completed,lora,2.0,4.0,42,256,50,0.0002,8,...,0.025592,2116.963867,2238.796387,1037.480552,20.749611,50,1.830166,1.715161,5.557572,None
2,lora_r4,completed,lora,4.0,8.0,42,256,50,0.0002,8,...,0.051172,2117.500977,2240.944824,1039.410338,20.788207,50,1.814759,1.670826,5.316560,None
3,lora_r8,completed,lora,8.0,16.0,42,256,50,0.0002,8,...,0.102291,2118.575195,2245.241699,1038.928935,20.778579,50,1.801170,1.637698,5.143314,None
4,lora_r16,completed,lora,16.0,32.0,42,256,50,0.0002,8,...,0.204372,2120.723633,2253.835449,1040.318017,20.806360,50,1.787818,1.614362,5.024684,None


In [5]:
summary_df[[
    "run_name",
    "status",
    "method",
    "r",
    "trainable_params",
    "trainable_ratio",
    "peak_gpu_memory_mb",
    "eval_loss",
    "perplexity",
    "step_time_sec",
]].sort_values(["method", "r"], na_position="first")


,run_name,status,method,r,trainable_params,trainable_ratio,peak_gpu_memory_mb,eval_loss,perplexity,step_time_sec
0,full_ft_baseline,completed,full_ft,NaN,1100048384,100.000000,8586.855469,2.017092,7.516432,21.800695
1,lora_r2,completed,lora,2.0,281600,0.025592,2238.796387,1.715161,5.557572,20.749611
2,lora_r4,completed,lora,4.0,563200,0.051172,2240.944824,1.670826,5.316560,20.788207
3,lora_r8,completed,lora,8.0,1126400,0.102291,2245.241699,1.637698,5.143314,20.778579
4,lora_r16,completed,lora,16.0,2252800,0.204372,2253.835449,1.614362,5.024684,20.806360
